# 01 - EDA (Phase 1C)
Chạy sau `python -m src.data.dedup`. Notebook chỉ gọi hàm trong `src/data/eda.py`; mọi nhận xét phải dựa trên số hiện ra ở đây.

In [1]:
import sys, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.common.config import load_config
from src.common.paths import resolve
from src.data.eda import eda_summary, make_figures
dcfg, fcfg = load_config('data'), load_config('features')
df = pd.read_parquet(resolve(dcfg['interim']['tracks_clean']))
pairs = pd.read_csv(resolve(dcfg['interim']['near_pairs']))
df.head()

,track_id,artist_id,artist_name,album_id,title,genre,license_raw,duration,subset,license_family,file_ok,drop_reason,duration_s,file_sha1,exact_dup_of,dup_group,nn_cosine
0,2,1,AWOL,1,Food,Hip-Hop,Attribution-NonCommercial-ShareAlike 3.0 Inter...,168,small,unknown,True,,29.98,6976e70271234351250ada3fc49c5afd69ba9864,0.0,2,0.946232
1,5,1,AWOL,1,This World,Hip-Hop,Attribution-NonCommercial-ShareAlike 3.0 Inter...,206,small,unknown,True,,30.00,d25363d1167965f376a0e837fa5faa662598cfe5,0.0,5,0.942061
2,10,6,Kurt Vile,6,Freeway,Pop,Attribution-NonCommercial-NoDerivatives (aka M...,161,small,unknown,True,,29.98,0efc06eab57f36065d7ccff153db8fa2d16df28f,0.0,10,0.951300
3,140,54,Alec K. Redfearn & the Eyesores,61,Queen Of The Wires,Folk,Attribution-Noncommercial-No Derivative Works ...,253,small,unknown,True,,29.98,208851706e7fe3b7c1a679cfb76f2139f205c943,0.0,140,0.986577
4,141,54,Alec K. Redfearn & the Eyesores,60,Ohio,Folk,Attribution-Noncommercial-No Derivative Works ...,182,small,unknown,True,,29.98,f19f507137614cdedee99d297848a7ba553c36ae,0.0,141,0.941406


In [2]:
summ = eda_summary(df, pairs)
summ

{'n_tracks_metadata': 8000,
 'n_tracks_ok': 7992,
 'drop_reason_counts': {'too_short': 3, 'decode_error': 3, 'silent': 2},
 'n_artists_ok': 2307,
 'n_albums_ok': 2462,
 'tracks_per_artist': {'min': 1, 'median': 1.0, 'max': 208},
 'genre_counts': {'Pop': 1000,
  'International': 1000,
  'Folk': 999,
  'Experimental': 999,
  'Rock': 999,
  'Electronic': 999,
  'Instrumental': 999,
  'Hip-Hop': 997},
 'license_family_counts': {'unknown': 7992},
 'duration_sec': {'min': 29.96, 'median': 29.98, 'max': 30.0},
 'n_dup_groups_gt1': 47,
 'near_pairs_flagged': 77,
 'near_pairs_same_artist': 60,
 'nn_cosine_quantiles': {'0.5': 0.9263157248497009,
  '0.9': 0.9832604825496674,
  '0.99': 0.9968502467870712,
  '1.0': 1.0000001192092896}}

In [3]:
make_figures(df, resolve(fcfg['paths']['logmel_dir']), resolve(dcfg['interim']['figures_dir']), resolve(dcfg['interim']['tables_dir']), seed=dcfg['seed'])

[WindowsPath('C:/Users/Admin/Downloads/SIC/PROJECT/PHASE1/music-copyright-ai/reports/figures/eda_genre.png'),
 WindowsPath('C:/Users/Admin/Downloads/SIC/PROJECT/PHASE1/music-copyright-ai/reports/figures/eda_license_family.png'),
 WindowsPath('C:/Users/Admin/Downloads/SIC/PROJECT/PHASE1/music-copyright-ai/reports/figures/eda_tracks_per_artist.png'),
 WindowsPath('C:/Users/Admin/Downloads/SIC/PROJECT/PHASE1/music-copyright-ai/reports/figures/eda_duration.png'),
 WindowsPath('C:/Users/Admin/Downloads/SIC/PROJECT/PHASE1/music-copyright-ai/reports/figures/eda_nn_cosine.png'),
 WindowsPath('C:/Users/Admin/Downloads/SIC/PROJECT/PHASE1/music-copyright-ai/reports/figures/eda_logmel_examples.png')]

## Hiệu chỉnh ngưỡng nghi trùng
Xem histogram `eda_nn_cosine.png` và phân vị `nn_cosine_quantiles`. Ngưỡng 0.995 trong `configs/data.yaml` là đề xuất; kiểm tra tay vài cặp trong `near_dup_pairs.csv` (đặc biệt cặp khác nghệ sĩ) trước khi giữ hay đổi.

In [4]:
pairs.sort_values('cosine', ascending=False).head(20)

,track_a,track_b,cosine,same_artist
67,120470,133016,1.000000,True
22,40541,40542,1.000000,True
70,130170,130650,1.000000,True
59,114222,115292,1.000000,True
60,114223,115290,1.000000,True
73,144933,145702,1.000000,True
32,57371,72149,1.000000,True
39,74348,86761,1.000000,True
62,114236,125820,1.000000,True
28,45102,51785,1.000000,True


## Nhận xét (điền sau khi có số)
- Phân bố thể loại / họ giấy phép:
- Nghệ sĩ có quá ít/nhiều bài:
- File bị loại và lý do:
- Quyết định về ngưỡng nghi trùng:
